# ED Pipeline v8 — Phase 1 (v6 tracker schema) + Phase 2 Guarded
This notebook keeps Phase 1 stable with **v6 tracker schema verbatim** (`equip_id, name, location, status, last_seen, battery, confidence`) and switches in-notebook `tracker_core` to that schema using pandas `.loc`. Phase 2 remains guarded.


In [ ]:
# === Cell 0: Canonical module bind (no side effects) ===
from pathlib import Path
import runpy
mod=None
if Path('./ed_pipeline_v8.py').exists():
    try:
        mod=runpy.run_path('./ed_pipeline_v8.py')
    except Exception as e:
        print('Note: ed_pipeline_v8.py present but could not be run via runpy:', e)
if mod is None and Path('./ed_pipeline_v8(2).py').exists():
    mod=runpy.run_path('./ed_pipeline_v8(2).py')
if mod is not None:
    WorkflowState=mod['WorkflowState']; TinyCritics=mod['TinyCritics']; CONFIG=mod['CONFIG']
assert 'WorkflowState' in globals() and 'TinyCritics' in globals() and 'CONFIG' in globals(), 'Missing core symbols.'
print('OK: canonical symbols bound.')


In [ ]:
# === Cell A: Back-compat shim for WorkflowState.update_state_from_event ===
_WS=globals().get('WorkflowState'); assert _WS is not None
if not hasattr(_WS,'update_state_from_event'):
    def _update_state_from_event(self,event:dict):
        for candidate in ('apply_event','update_from_event','update_from_dict','update'):
            fn=getattr(self,candidate,None)
            if callable(fn): return fn(event)
        backing=getattr(self,'state',None)
        if backing is None: backing={}; setattr(self,'state',backing)
        for k,v in (event or {}).items():
            parts=str(k).split('.')
            d=backing
            for p in parts[:-1]:
                if p not in d or not isinstance(d[p],dict): d[p]={}
                d=d[p]
            d[parts[-1]]=v
        return self
    setattr(_WS,'update_state_from_event',_update_state_from_event)
_=_WS(role='nurse').update_state_from_event({'age':60,'vitals.sbp':95}); print('OK: update_state_from_event present')


In [ ]:
# === Cell T(v6): tracker_core in-notebook using v6 schema (all .loc) ===
import sys, types, pandas as pd, numpy as np
from pathlib import Path

assert 'CONFIG' in globals(), 'CONFIG must be available'

def _cfg(CONFIG, key, default=None):
    try:
        return CONFIG.get(key, default)
    except Exception:
        return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default

def _ensure_parent(p: Path):
    p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)

def _utcnow_iso():
    return pd.Timestamp.utcnow().isoformat()

class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv = Path(status_csv); _ensure_parent(self.status_csv)
        if not self.status_csv.exists():
            cols=['equip_id','name','location','status','last_seen','battery','confidence']
            pd.DataFrame(columns=cols).to_csv(self.status_csv, index=False)
    def read(self) -> pd.DataFrame:
        try:
            df = pd.read_csv(self.status_csv)
        except Exception:
            df = pd.DataFrame(columns=['equip_id','name','location','status','last_seen','battery','confidence'])
        # Ensure required columns; preserve order
        for c in ['equip_id','name','location','status','last_seen','battery','confidence']:
            if c not in df.columns:
                df[c] = np.nan
        # dtypes: keep flexible; ensure equip_id as str for joins
        df['equip_id'] = df['equip_id'].astype(str)
        return df[['equip_id','name','location','status','last_seen','battery','confidence']]
    def upsert(self, rec: dict) -> None:
        df = self.read()
        # rec is a dict with v6 keys
        eqid = str(rec.get('equip_id',''))
        if (df['equip_id'] == eqid).any():
            idx = df.index[df['equip_id'] == eqid][0]
            for k in ['name','location','status','last_seen','battery','confidence']:
                if k in df.columns:
                    df.loc[idx, k] = rec.get(k, df.loc[idx, k])
        else:
            df = pd.concat([df, pd.DataFrame([rec])], ignore_index=True)
        df.to_csv(self.status_csv, index=False)

class MovesLogRepository:
    def __init__(self, moves_csv: Path):
        self.moves_csv = Path(moves_csv); _ensure_parent(self.moves_csv)
        if not self.moves_csv.exists():
            pd.DataFrame(columns=['equip_id','from','to','ts']).to_csv(self.moves_csv, index=False)
    def append(self, equip_id: str, loc_from: str, loc_to: str, ts_iso: str) -> None:
        row = pd.DataFrame([{'equip_id': str(equip_id), 'from': loc_from, 'to': loc_to, 'ts': ts_iso}])
        try:
            prev = pd.read_csv(self.moves_csv)
            df = pd.concat([prev, row], ignore_index=True)
        except Exception:
            df = row
        df.to_csv(self.moves_csv, index=False)
    def read(self) -> pd.DataFrame:
        try:
            return pd.read_csv(self.moves_csv)
        except Exception:
            return pd.DataFrame(columns=['equip_id','from','to','ts'])

class SOPRegistry:
    def __init__(self, sop_csv: Path):
        self.sop_csv = Path(sop_csv); _ensure_parent(self.sop_csv)
        if not self.sop_csv.exists():
            pd.DataFrame([
                {'sop_id':'sop-triage','title':'ED Triage','url':'about:blank','status':'active'},
                {'sop_id':'sop-ecg','title':'ECG Acquisition','url':'about:blank','status':'active'},
                {'sop_id':'sop-sepsis','title':'Sepsis Bundle','url':'about:blank','status':'active'},
                {'sop_id':'sop-stemi','title':'STEMI Activation','url':'about:blank','status':'active'},
                {'sop_id':'sop-airway','title':'Difficult Airway','url':'about:blank','status':'active'},
            ]).to_csv(self.sop_csv, index=False)
    def read(self) -> pd.DataFrame | None:
        try:
            return pd.read_csv(self.sop_csv)
        except Exception:
            return None

class QRService:
    def __init__(self, out_dir: Path):
        self.out_dir = Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self, payload: str) -> str:
        try:
            import qrcode
            img = qrcode.make(payload)
            p = self.out_dir / f"qr_{int(pd.Timestamp.utcnow().timestamp())}.png"
            img.save(p); return str(p)
        except Exception:
            p = self.out_dir / f"qr_{int(pd.Timestamp.utcnow().timestamp())}.txt"
            p.write_text(payload); return str(p)
    def decode(self, path: str) -> str | None:
        try:
            from PIL import Image
            from pyzbar.pyzbar import decode as _decode
            res = _decode(Image.open(path))
            if res: return res[0].data.decode('utf-8', errors='ignore')
        except Exception:
            pass
        try:
            p = Path(path)
            if p.suffix.lower()=='.txt': return p.read_text()
        except Exception:
            pass
        return None

class TrackerService:
    def __init__(self, equipment_repo: 'EquipmentRepository', moves_repo: 'MovesLogRepository', sop_registry: 'SOPRegistry', qr: 'QRService', config):
        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config
    @classmethod
    def from_config(cls, CONFIG):
        return cls(
            EquipmentRepository(Path(_cfg(CONFIG,'EQUIPMENT_STATUS_PATH'))),
            MovesLogRepository(Path(_cfg(CONFIG,'EQUIPMENT_MOVES_LOG_PATH'))),
            SOPRegistry(Path(_cfg(CONFIG,'SOP_REGISTRY_PATH'))),
            QRService(Path(_cfg(CONFIG,'QR_OUTPUT_DIR'))), CONFIG)
    def equipment_status(self) -> pd.DataFrame:
        return self.equipment_repo.read()
    def log_move(self, equip_id: str, loc_from: str, loc_to: str) -> None:
        ts_iso = _utcnow_iso(); df = self.equipment_repo.read()
        name = ''
        if 'name' in df.columns and (df['equip_id'].astype(str)==str(equip_id)).any():
            name = df.loc[df['equip_id'].astype(str)==str(equip_id), 'name'].iloc[0]
        rec = {
            'equip_id': str(equip_id), 'name': name, 'location': loc_to,
            'status': 'moved', 'last_seen': ts_iso, 'battery': np.nan, 'confidence': np.nan
        }
        self.equipment_repo.upsert(rec)
        self.moves_repo.append(str(equip_id), loc_from or '', loc_to, ts_iso)
    def moves_summary(self) -> dict:
        log = self.moves_repo.read()
        if log.empty:
            return {'moves_per_equipment': log, 'routes': log}
        per_eq = log.groupby('equip_id').size().reset_index(name='moves').sort_values('moves', ascending=False)
        routes = log.groupby(['from','to']).size().reset_index(name='count').sort_values('count', ascending=False)
        return {'moves_per_equipment': per_eq, 'routes': routes}
    def sop_table(self) -> pd.DataFrame:
        return self.sop_registry.read()
    def search_sop(self, query: str) -> pd.DataFrame:
        df = self.sop_registry.read().copy(); q = (query or '').strip().lower()
        if df is None or df.empty or not q:
            return df
        cols = [c for c in ['sop_id','title','keywords','version','status'] if c in df.columns]
        mask = df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)
        return df[mask]
    def make_qr(self, payload: str) -> str:
        return self.qr.make(payload)
    def decode_qr_bytes(self, image_bytes: bytes):
        return self.qr.decode(image_bytes)

# Register as importable module name
import types as _types, sys as _sys
_tracker_mod = _types.ModuleType('tracker_core')
for _name, _obj in {
    'EquipmentRepository': EquipmentRepository,
    'MovesLogRepository': MovesLogRepository,
    'SOPRegistry': SOPRegistry,
    'QRService': QRService,
    'TrackerService': TrackerService,
}.items(): setattr(_tracker_mod,_name,_obj)
_sys.modules['tracker_core'] = _tracker_mod
print('OK: tracker_core (v6 schema) ready.')


In [ ]:
# === Cell C: CONFIG defaults per checklist ===
CONFIG['RUN_UI']=False; CONFIG['RUN_PIPELINE']=False
print('OK: CONFIG flags set (RUN_UI=False, RUN_PIPELINE=False)')


In [ ]:
# === Cell D: Delivery checklist (2–14) updated for v6 schema ===
import pandas as pd
from pathlib import Path
from tracker_core import TrackerService, QRService, EquipmentRepository, MovesLogRepository, SOPRegistry

# 2–6: TinyCritics cold-start
s=WorkflowState(role='nurse')
getattr(s,'touch_now',lambda *_:None)(pd.Timestamp.utcnow())
tc=TinyCritics(); p,b,u=tc.score(s,[{'id':'reassess_vitals','label':'Reassess vitals'},{'id':'order_ecg','label':'Order ECG'}])
assert len(p)==2 and (0<=p).all() and (p<=1).all()

# 7–14: Tracker core basic IO with v6 schema
t=TrackerService.from_config(CONFIG)
status_before=t.equipment_status()
t.log_move('pump-001','A1','B2')
status_after=t.equipment_status()
assert Path(CONFIG['EQUIPMENT_MOVES_LOG_PATH']).exists()
# Schema checks: required columns present
for c in ['equip_id','name','location','status','last_seen','battery','confidence']:
    assert c in status_after.columns, f'missing column {c}'

q = QRService(CONFIG['QR_OUTPUT_DIR']).make('poctest')
assert isinstance(q,str) and len(q)>0
sop = SOPRegistry(CONFIG['SOP_REGISTRY_PATH']).read()
assert sop is not None
print('SMOKE_OK')


In [ ]:
# === Cell E: Surfaces (present & offline-safe) ===
from pathlib import Path
from typing import Any, Dict
def refresh_sop_registry(CONFIG: Any, base_url: str='https://sop-notaufnahme.de/sop/') -> Dict[str,Any]:
    out_csv=Path(CONFIG['SOP_REGISTRY_PATH'])
    try:
        import requests; from bs4 import BeautifulSoup
    except Exception as e:
        return {'found':0,'saved':0,'errors':1,'error':f'missing libs: {e}'}
    # Offline-safe surface: return a summary; real save guarded by IO and network
    try:
        r = requests.get(base_url, timeout=15); r.raise_for_status()
        html = r.text
        return {'found_hint': html[:100], 'saved':0, 'errors':0, 'csv': str(out_csv)}
    except Exception as e:
        return {'found':0,'saved':0,'errors':1,'error':str(e),'csv':str(out_csv)}

def qr_scan_fallback(payload: str, tracker: 'TrackerService'):
    try:
        parts = dict(kv.split('=',1) for kv in payload.split('&') if '=' in kv)
        eq_id, to_loc = parts.get('id') or parts.get('equip_id'), parts.get('to')
        if eq_id and to_loc:
            tracker.log_move(eq_id, '', to_loc)
            return {'ok': True, 'equip_id': eq_id, 'to': to_loc}
        return {'ok': False, 'reason': 'missing equip_id/to'}
    except Exception as e:
        return {'ok': False, 'reason': str(e)}

ALERT_THRESHOLDS_MIN={'equipment_overdue':60,'lingering_patient':120}
print('OK: surfaces present.')


In [ ]:
# === Cell P2: Phase 2 guard scaffold ===
if CONFIG.get('RUN_PIPELINE'):
    print('Phase 2 enabled — add your guarded asserts here.')
else:
    print("Phase 2 disabled (set CONFIG['RUN_PIPELINE']=True to enable).")
